# MiniMax H3 リメイク計画

既存の [minimax_h3_comfyui_colab.ipynb](https://colab.research.google.com/drive/1Adu5cv3GmxN5_LZZ4cu6aZVUBsaY-v3H) の更新。Comfy の起動はそのノートのまま。このノートは計画 JSON を書くところまで。

- 本体は公式 R2V の `minimax_h3_ref2va_pruned_int8_convrot.safetensors`
- ワークフローは `video_minimax_h3_r2v.json` の `MiniMaxH3ReferenceToVideo`
- steps 20、4step LoRA はオフ
- `generate` は false。このノートから生成キューは投げない
- 動きの文が空の範囲は `blocked`。文は新しく書かない


In [ ]:
#@title 1) Drive をマウント
from google.colab import drive
drive.mount("/content/drive")
print("mounted")


In [ ]:
#@title 2) 計画だけ書く
SHEET = ""  #@param {type:"string"}
OUT = ""  #@param {type:"string"}

import importlib.util
import json
from pathlib import Path

MODULE = Path("/content/drive/MyDrive/Colab Notebooks/h3_live_remake.py")
if not MODULE.is_file():
    raise SystemExit(f"モジュールが無い: {MODULE}")
if not SHEET:
    raise SystemExit("SHEET が空。シート json のパスを入れて、このセルだけ再実行する。")

spec = importlib.util.spec_from_file_location("h3_live_remake", MODULE)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
plan = module.build_remake_plan(module.load_sheet(SHEET))
if plan["generate"] or plan["publish"] or plan["comfy"]["queue"]:
    raise SystemExit("計画が生成を要求しているので止めた")

out = Path(OUT) if OUT else Path(SHEET).with_name(Path(SHEET).stem + "-plan.json")
out.write_text(json.dumps(plan, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(plan["status"], "spans", len(plan["spans"]), "steps", plan["model"]["steps"])
print("unet", plan["comfy"]["unet"])
print("WROTE", out)
for item in plan["blocked"]:
    print("-", item)
print("生成ボタンは押していない。")


## 重みが無いときだけ

既存ノートの初期 MODE は `t2v_i2v` で、動画 VAE は fp16 を落とす。公式 R2V が読むのは `minimax_h3_ref2va_pruned_int8_convrot.safetensors` と `minimax_h3_video_vae_int8_convrot.safetensors`。無いファイルだけ次のセルが足す。既定では走らない。


In [ ]:
#@title 3) 足りない R2V の重みだけ落とす
RUN_DOWNLOAD = False  #@param {type:"boolean"}
import os
import subprocess
import sys

if not RUN_DOWNLOAD:
    raise SystemExit("RUN_DOWNLOAD が false。計画セルだけで足りる。")

COMFY_DIR = "/content/ComfyUI"
HF = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main"
files = [
    "diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors",
    "text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
    "vae/minimax_h3_video_vae_int8_convrot.safetensors",
    "vae/minimax_h3_audio_vae_fp32.safetensors",
]
for rel in files:
    dest = os.path.join(COMFY_DIR, "models", rel)
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    if os.path.isfile(dest) and os.path.getsize(dest) > 1_000_000:
        print("skip", os.path.basename(dest))
        continue
    url = f"{HF}/{rel}"
    print("downloading", os.path.basename(dest))
    subprocess.run(["wget", "-c", "--show-progress", "-O", dest, url], check=True)
    if os.path.getsize(dest) < 1_000_000:
        sys.exit(f"DL 失敗: {dest}")
print("weights ready")


In [ ]:
#@title 4) 公式 R2V ワークフローを置く（生成はしない）
import os
import subprocess

COMFY_DIR = "/content/ComfyUI"
wf_dir = f"{COMFY_DIR}/user/default/workflows"
os.makedirs(wf_dir, exist_ok=True)
name = "video_minimax_h3_r2v.json"
dest = f"{wf_dir}/{name}"
url = "https://raw.githubusercontent.com/Comfy-Org/workflow_templates/main/templates/" + name
subprocess.run(["wget", "-q", "-O", dest, url], check=True)
print("saved", dest)
print("UI は既存ノートの起動セルで開く。プロンプトを貼ったあと、生成ボタンは人間が押す。")
